# 🌀 Inception (GoogLeNet) — Notes + Interview
---
> **Simple English** | **Interview Ready** | Year: 2014 | Creator: Google

## 📌 What is Inception? (Simple English)
- Inception = **multiple filter sizes in parallel** in the same layer
- Key question it answers: "What's the right filter size — 1×1? 3×3? 5×5?"
- Inception answer: **use all of them at once!** → concatenate results
- Also uses **1×1 convolutions** to reduce computation (bottleneck)
- GoogLeNet won ImageNet 2014 with only 6.7M params (vs VGG's 138M!)

## 🔑 Inception Module
```
Input x
  ├── 1×1 Conv
  ├── 1×1 Conv → 3×3 Conv
  ├── 1×1 Conv → 5×5 Conv
  └── MaxPool  → 1×1 Conv
  ↓
Concatenate all outputs (along depth)
```

## 🧱 Why 1×1 Convolution?
- 1×1 conv = **dimension reduction** (reduces channels before expensive 3×3 or 5×5)
- Reduces computation dramatically
- Also adds non-linearity
- Example: 256 channels → 1×1(64) → 3×3(128) → saves 88% computation!

In [ ]:
import tensorflow as tf

# ── Inception Module ──
def inception_module(x, f1, f2_reduce, f2, f3_reduce, f3, f_pool):
    # Branch 1: 1×1 conv
    b1 = tf.keras.layers.Conv2D(f1,   (1,1), activation='relu', padding='same')(x)

    # Branch 2: 1×1 reduce → 3×3
    b2 = tf.keras.layers.Conv2D(f2_reduce,(1,1),activation='relu',padding='same')(x)
    b2 = tf.keras.layers.Conv2D(f2,   (3,3), activation='relu', padding='same')(b2)

    # Branch 3: 1×1 reduce → 5×5
    b3 = tf.keras.layers.Conv2D(f3_reduce,(1,1),activation='relu',padding='same')(x)
    b3 = tf.keras.layers.Conv2D(f3,   (5,5), activation='relu', padding='same')(b3)

    # Branch 4: MaxPool → 1×1 proj
    b4 = tf.keras.layers.MaxPooling2D(3,strides=1,padding='same')(x)
    b4 = tf.keras.layers.Conv2D(f_pool,(1,1),activation='relu',padding='same')(b4)

    # Concatenate all branches along depth axis
    out = tf.keras.layers.Concatenate(axis=-1)([b1, b2, b3, b4])
    return out

# Build small Inception network
inputs = tf.keras.Input(shape=(28,28,1))
x = tf.keras.layers.Conv2D(32,(3,3),activation='relu',padding='same')(inputs)
x = tf.keras.layers.MaxPooling2D(2,2)(x)

# Inception module: params = (f1, f2_reduce, f2, f3_reduce, f3, f_pool)
x = inception_module(x, 32, 16, 32, 8, 16, 16)    # out: 96 channels
x = inception_module(x, 64, 32, 64, 16, 32, 32)   # out: 192 channels

x = tf.keras.layers.GlobalAveragePooling2D()(x)
output = tf.keras.layers.Dense(10, activation='softmax')(x)

inception_model = tf.keras.Model(inputs, output, name='Inception-Custom')
inception_model.summary()
print(f"\nTotal params: {inception_model.count_params():,}")

In [ ]:
# Pretrained InceptionV3
inception_v3 = tf.keras.applications.InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(299,299,3)   # InceptionV3 uses 299×299
)
print(f"InceptionV3 layers: {len(inception_v3.layers)}")
print(f"InceptionV3 params: {inception_v3.count_params():,}  (~24M)")

# Transfer learning
inception_v3.trainable = False
x = inception_v3.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256,activation='relu')(x)
out = tf.keras.layers.Dense(10,activation='softmax')(x)
model_iv3 = tf.keras.Model(inception_v3.input, out)
print(f"\nTrainable params: {sum([tf.size(w).numpy() for w in model_iv3.trainable_weights]):,}")

## 🗣️ Interview Q&A

**Q: What is the Inception module?**
> A block that applies multiple convolutions (1×1, 3×3, 5×5) and max pooling IN PARALLEL on the same input, then concatenates all outputs. This lets the network decide which scale of features matters most.

**Q: What is the role of 1×1 convolution in Inception?**
> Acts as a bottleneck — reduces the number of channels (depth) before expensive 3×3 or 5×5 convolutions. Massively reduces computation without losing much information.

**Q: How does Inception compare to VGG in parameters?**
> GoogLeNet: ~6.7M parameters. VGG16: ~138M parameters. Inception is 20× smaller but achieved similar/better accuracy on ImageNet.

**Q: What is InceptionV3?**
> Improved version with factorized convolutions (5×5 → two 3×3, 7×7 → 1×7 + 7×1). Also uses auxiliary classifiers during training for gradient flow. Input: 299×299.